In [90]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import re

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import data_tools as dt
import writing_tools as wt
from utils import parse_casenum

from matplotlib import pyplot as plt
from sklearn.linear_model import LogisticRegression
from IPython.core.display import HTML
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import statsmodels.api as sm
from stargazer.stargazer import Stargazer

from scipy.spatial.distance import mahalanobis


plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

os.environ['LOKY_MAX_CPU_COUNT'] = '1' # because of windows core count warning

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10



In [91]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data_w_embeddings.parquet"))

In [92]:
# Council district dummies

vals = ['1','2','3','4','5','6','7','8','9','10','11','12','13','14','15','CITYWIDE']
for v in vals:
    df[f'cd_{v}'] = 0

invalids = []
for idx, row in df.iterrows():
    splitted = re.split(r"[,;]", row['council_district'])
    council_districts = [x.strip() for x in splitted if len(x.strip())>0]
    for cd in council_districts:
        if cd in vals:
            df.loc[idx, f'cd_{cd}'] = 1
        else:
            invalids.append((idx, cd))

print(invalids)

# apply fixes
df.loc[82, "cd_9"] = 1
df.loc[572, "cd_CITYWIDE"] = 1

[(82, 'Multiple'), (571, 'ALL')]


In [93]:
# Suffix encoding

sfx_df = pd.read_csv(os.path.join(LOCAL_PATH, "intermediate_data/cpc", "suffix_groups.csv"))
sfx_df = sfx_df.set_index("suffix")
groups = sfx_df['group'].unique().tolist()
for grp in groups:
    df[f"sfx_{grp}"] = 0

for idx, row in df.iterrows():
    title = row["title"]
    parsed_casenum = parse_casenum(title)
    suffixes = parsed_casenum['suffixes']
    for sfx in suffixes:
        grp = sfx_df.loc[sfx, 'group']
        df.at[idx, f"sfx_{grp}"] = 1


In [94]:
# Agenda order and number of agenda items

df["agenda_order"] = np.nan
for idx, row in df.iterrows():
    item_no = row["item_no"]
    agenda_order = int(re.match(r'\d+', item_no).group())
    df.at[idx, "agenda_order"] = agenda_order

df["num_agenda_items"] = df.groupby("date")["agenda_order"].transform("max")

In [95]:
# support and opposition

df["log2_support"] = np.log2(df["n_support"] + 1)
df["log2_oppose"] = np.log2(df["n_oppose"] + 1)

In [96]:
# Calculate mahalanobis distance

cols = [f'd{i}' for i in range(N_COMPONENTS)]
centroid_cols = [f'cluster_d{i}' for i in range(N_COMPONENTS)]

cov_invs = {}
for cluster, group in df.groupby('cluster'):
    cov = group[cols].cov().values
    cov_invs[cluster] = np.linalg.inv(cov)

df['mahalanobis'] = np.nan
for idx, row in df.iterrows():
    cluster = row['cluster']
    x = row[cols].values
    centroid = row[centroid_cols].values
    df.at[idx, 'mahalanobis'] = mahalanobis(x, centroid, cov_invs[cluster])


In [97]:
# Outcome variable

df["project_result"].value_counts()

df["outcome"] = 1
df.loc[df["project_result"] == "APPROVED", "outcome"] = 2
df.loc[df["project_result"].isin(['DELAYED', 'DENIED', 'APPLICATION WIDTHDRAWN']), "outcome"] = 0


In [98]:
# Output regression data
df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_data.parquet"))